In [1]:
# Import the libraries:
# requests downloads the web page, BeautifulSoup reads the HTML, pandas makes the table
import requests
from bs4 import BeautifulSoup
import pandas as pd

In [2]:
# Download the article. The User-Agent makes the request look like it comes from a normal browser
url = "https://www.tagesschau.de/ausland/mexiko-feierlichkeiten-november-100.html"
response = requests.get(url, headers={"User-Agent": "Mozilla/5.0"})

# 200 means the page was downloaded successfully
print(response.status_code)

200


In [3]:
# Check that we got the real article and not a cookie or error page
print("Día de Muertos" in response.text)

True


In [4]:
# Turn the HTML into an object we can search
soup = BeautifulSoup(response.text, "html.parser")

In [5]:
# Title: the h1 contains the topline and the headline together,

title = soup.find("h1").get_text(strip=True)
print(title)

Día de MuertosBunte Toten-Parade zieht durch Mexiko-Stadt


In [6]:
# so I take only the span that holds the headline
title = soup.find("span", class_="article-head__headline--text").get_text(strip=True)
print(title)

Bunte Toten-Parade zieht durch Mexiko-Stadt


In [7]:
# Publication time: the page has no <time> tag,
# the date is in the element with the class "metatextline"
timestamp = soup.find(class_="metatextline").get_text(strip=True)
print(timestamp)

Stand: 01.11.2026 • 08:57 Uhr


In [8]:
# Remove the word "Stand:" so only the date and time are left
timestamp = timestamp.replace("Stand:", "").strip()
print(timestamp)

01.11.2026 • 08:57 Uhr


In [9]:
# Author: found the class with Inspect in the browser
author = soup.find(class_="article-head__topline").get_text(strip=True)
print(author)

Día de Muertos


In [10]:
# Main text: every paragraph of the article has the class "textabsatz",
# so find_all collects all of them and join puts them together with line breaks
paragraphs = soup.find_all("p", class_="textabsatz m-ten m-offset-one l-eight l-offset-two columns twelve")
main_text = "\n".join(p.get_text(strip=True) for p in paragraphs)
print(main_text)

Anlässlich des "Tages der Toten" sind Tausende kostümierte Menschen durch die Innenstadt von Mexiko-Stadt gezogen. Sie wurden von festlich geschmückten Wagen und Musikgruppen begleitet. Nach Angaben der Stadtverwaltung verfolgten Hunderttausende das gruselig-schöne Geschehen als Zuschauer.
Auf dem zentralen Zócalo-Platz wurde eine elf Stockwerke hohe "La Catrina" aufgestellt - die elegant gekleidete weibliche Skelettfigur mit Hut ist eine Personifikation des Todes.
Die Straßenparade in der mexikanischen Hauptstadt entstand erst vor wenigen Jahren als Touristenattraktion. Sie wurde von einer Szene aus dem James-Bond-Film "Spectre" inspiriert. In einer Szene läuft der britische Agent durch einen solchen Umzug in Mexiko-Stadt - dabei gab es bis dahin gar keine Parade. "Alle fragten mich: 'Und die Parade? Wann ist die Parade?'", sagte der frühere Bürgermeister Miguel Ángel Mancera, der den ersten Umzug 2016 organisiert hatte, der Nachrichtenagentur AFP.
Auch der Pixar-Film "Coco" trug dazu

In [11]:
# Put all four fields into a table with one row
df = pd.DataFrame([{
    "Publication timestamp": timestamp,
    "Author": author,
    "Title": title,
    "Main Text": main_text,
}])
df

,Publication timestamp,Author,Title,Main Text
0,01.11.2026 • 08:57 Uhr,Día de Muertos,Bunte Toten-Parade zieht durch Mexiko-Stadt,"Anlässlich des ""Tages der Toten"" sind Tausende..."


In [12]:
# Print the full main text to check nothing is missing (the table cuts it off)
print(df["Main Text"][0])

Anlässlich des "Tages der Toten" sind Tausende kostümierte Menschen durch die Innenstadt von Mexiko-Stadt gezogen. Sie wurden von festlich geschmückten Wagen und Musikgruppen begleitet. Nach Angaben der Stadtverwaltung verfolgten Hunderttausende das gruselig-schöne Geschehen als Zuschauer.
Auf dem zentralen Zócalo-Platz wurde eine elf Stockwerke hohe "La Catrina" aufgestellt - die elegant gekleidete weibliche Skelettfigur mit Hut ist eine Personifikation des Todes.
Die Straßenparade in der mexikanischen Hauptstadt entstand erst vor wenigen Jahren als Touristenattraktion. Sie wurde von einer Szene aus dem James-Bond-Film "Spectre" inspiriert. In einer Szene läuft der britische Agent durch einen solchen Umzug in Mexiko-Stadt - dabei gab es bis dahin gar keine Parade. "Alle fragten mich: 'Und die Parade? Wann ist die Parade?'", sagte der frühere Bürgermeister Miguel Ángel Mancera, der den ersten Umzug 2016 organisiert hatte, der Nachrichtenagentur AFP.
Auch der Pixar-Film "Coco" trug dazu

In [13]:
# Export as csv
df.to_csv("article.csv", index=False)